# ForgetProof — 02: Unlearn + audit

Runs all baseline unlearning methods (Gradient Ascent, Gradient Diff, NPO) plus
our proposed method (RC-NPO), then runs the full audit suite against each
resulting model and saves results to `results/<method>.json`.

Run 01_setup_and_baseline.ipynb first so the target model is saved to Drive.

In [ ]:
!git clone https://github.com/vinay-reddaboina/forgetproof.git
%cd forgetproof
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import copy, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.data_utils import load_tofu, load_tofu_real_authors

device = "cuda" if torch.cuda.is_available() else "cpu"
TARGET_DIR = "/content/drive/MyDrive/forgetproof/target_model"

tokenizer = AutoTokenizer.from_pretrained(TARGET_DIR)
base_model = AutoModelForCausalLM.from_pretrained(TARGET_DIR, torch_dtype=torch.float16).to(device)

FORGET_SPLIT = "forget10"
forget_ds, retain_ds = load_tofu(FORGET_SPLIT)
unseen_ds = load_tofu_real_authors()
print("loaded target model + data")

In [ ]:
# Optional: paraphrased forget-set questions for the rephrase-extraction attack.
# TOFU's forget10_perturbed split has these; falls back to None (attack skipped)
# if it's not available for this split in the installed datasets version.

paraphrased_examples = None
try:
    from datasets import load_dataset
    perturbed = load_dataset("locuslab/TOFU", "forget10_perturbed")["train"]
    paraphrased_examples = [
        {"paraphrased_question": ex["paraphrased_question"], "answer": ex["answer"]}
        for ex in perturbed.select(range(min(50, len(perturbed))))
    ]
    print("loaded", len(paraphrased_examples), "paraphrased examples")
except Exception as e:
    print("no perturbed split available, skipping rephrase attack:", e)

In [ ]:
# Group forget-set questions by author/entity for the representation-probe attack.
# TOFU's 'author' field (if present) groups QA pairs by fictitious author.

entities_for_probe = None
if "author" in forget_ds.column_names:
    entities_for_probe = {}
    for ex in forget_ds:
        entities_for_probe.setdefault(ex["author"], []).append(ex["question"])
    # keep it small so the probe attack runs fast
    entities_for_probe = {k: v[:10] for k, v in list(entities_for_probe.items())[:10]}
    print("probe entities:", list(entities_for_probe.keys()))
else:
    print("no 'author' field found -- skipping probe attack (check dataset schema)")

In [ ]:
from src.unlearn.grad_ascent import unlearn_grad_ascent
from src.unlearn.grad_diff import unlearn_grad_diff
from src.unlearn.npo import unlearn_npo
from src.unlearn.improved import unlearn_rc_npo
from src.eval.run_eval import run_full_audit, save_results

METHODS = {
    "grad_ascent": lambda m: unlearn_grad_ascent(m, tokenizer, forget_ds, device=device),
    "grad_diff":   lambda m: unlearn_grad_diff(m, tokenizer, forget_ds, retain_ds, device=device),
    "npo":         lambda m: unlearn_npo(m, tokenizer, forget_ds, retain_ds, device=device),
    "rc_npo":      lambda m: unlearn_rc_npo(m, tokenizer, forget_ds, retain_ds, device=device),
}

all_results = {}
for name, unlearn_fn in METHODS.items():
    print(f"\n=== {name} ===")
    model_copy = copy.deepcopy(base_model)
    unlearned = unlearn_fn(model_copy)

    results = run_full_audit(
        unlearned, tokenizer, method_name=name,
        forget_ds=forget_ds, retain_ds=retain_ds, unseen_ds=unseen_ds,
        paraphrased_examples=paraphrased_examples,
        entities_for_probe=entities_for_probe,
        device=device,
    )
    save_results(results, f"results/{name}.json")
    all_results[name] = results

    # also checkpoint the unlearned model to Drive in case a later cell fails
    unlearned.save_pretrained(f"/content/drive/MyDrive/forgetproof/{name}_model")
    del model_copy, unlearned
    torch.cuda.empty_cache()

In [ ]:
# Quick comparison table -- headline numbers per method
import pandas as pd

rows = []
for name, r in all_results.items():
    rows.append({
        "method": name,
        "RTT recovery": r["relearning_attack"]["recovery_score"],
        "MIA leak": r["membership_inference"]["leak_score"],
        "rephrase leak rate": r.get("rephrase_extraction", {}).get("leak_rate"),
        "probe acc above chance": r.get("representation_probe", {}).get("above_chance"),
        "model utility loss": r["model_utility_loss"],
    })
pd.DataFrame(rows)

In [ ]:
# Push results json back to the repo (run `git config` with your identity first if needed)
!git add results/*.json
!git commit -m "Add audit results for grad_ascent, grad_diff, npo, rc_npo"
!git push